In [0]:
# Spark Structured Streaming — Intro
# Reading from a rate source (built-in generator)

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, FloatType

In [0]:
# 1. Create streaming source (rate = generates rows)
streaming_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)
print(f"Is streaming: {streaming_df.isStreaming}")
print(f"Schema: {streaming_df.schema}")

Is streaming: True
Schema: StructType([StructField('timestamp', TimestampType(), True), StructField('value', LongType(), True)])


In [0]:
# 2. Transform streaming data
processed = (streaming_df
    .withColumn("model_id",
        F.col("value") % 5)
    .withColumn("accuracy",
        70 + (F.col("value") % 30).cast("float"))
    .withColumn("timestamp_str",
        F.col("timestamp").cast("string"))
)

In [0]:

# 3. Write stream to memory (for testing)
query = (processed.writeStream
    .format("memory")
    .queryName("ml_stream")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/ml_checkpoints/ml_stream")
    .trigger(availableNow=True)
    .start()
)

# Wait 5 seconds then query
import time
time.sleep(5)

In [0]:
# 4. Query the stream like a regular table!
spark.sql("""
    SELECT model_id,
           COUNT(*) as predictions,
           ROUND(AVG(accuracy), 2) as avg_accuracy
    FROM ml_stream
    GROUP BY model_id
    ORDER BY model_id
""").show()

query.stop()
print("Streaming query stopped!")

+--------+-----------+------------+
|model_id|predictions|avg_accuracy|
+--------+-----------+------------+
|       0|          2|        72.5|
|       1|          2|        73.5|
|       2|          2|        74.5|
|       3|          2|        75.5|
|       4|          2|        76.5|
+--------+-----------+------------+

Streaming query stopped!
